# Relevancy Pipeline NLP - YouTube Comment: Benchmark & Evaluation
Project Capstone: Rifki Afina Putri, S.T., M.S., Ph.D.

Menilai **relevansi** komentar YouTube terhadap topik PLTU/PLTN di Indonesia (biner: Relevan / Tidak). Seragam dengan notebook Stance & Sentiment.

* **Input File Tunggal**: `relevance_input_yt.csv` (124 komentar gold + konteks video + kolom `v1_original`).
* **Model**: Google AI Studio API (`gemma-4-26b-a4b-it`), key dari **Colab Secrets** (`GEMMA`).
* **Ground Truth**: kolom `relevan` (1 = relevan, 0 = tidak).
* **3 Konfigurasi**: `v1_original` (prediksi model lama dari kolom gold, tanpa API), `v1_title` (prompt lama, judul saja), `v2_ctx` (prompt baru berbasis Pedoman v2 + konteks video penuh).
* **Stateful & Resumable**: progres tiap config disimpan ke `preds_relevan_<config>.csv`; jalankan ulang -> lanjut otomatis.
* **Metrik**: set gold **80% relevan** (dirancang seimbang per kelas stance), jadi **akurasi mentah menyesatkan**. Lihat **recall** (relevan tertangkap), **specificity** (tidak-relevan tertolak), dan **prec@20%/30%** (perkiraan presisi bila di data asli hanya 20%/30% komentar relevan).

## 1. Import Libraries & Muat Dataset Input
Upload `relevance_input_yt.csv` ke Colab, lalu jalankan sel ini.

In [ ]:
import pandas as pd
import numpy as np
import os, re, json, time, math

INPUT_FILE = 'relevance_input_yt.csv'
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"File '{INPUT_FILE}' tidak ada. Upload dulu ke Colab (ikon folder di kiri).")

df = pd.read_csv(INPUT_FILE, dtype=str, keep_default_na=False)
df.columns = df.columns.astype(str).str.strip()
df['i'] = df.index  # id stabil untuk checkpoint
y_true = (df['relevan'] == '1').tolist()
print(f'[ok] {len(df)} komentar dimuat | relevan={sum(y_true)} ({sum(y_true)/len(df):.0%}) | tidak={len(df)-sum(y_true)}')
print('kolom:', df.columns.tolist())
df[['i', 'target', 'relevan', 'v1_original', 'text']].head(5)

## 2. Set Up Google GenAI Client & API Key
API key diambil dari **Colab Secrets** bernama `GEMMA` (ikon kunci di kiri). Jangan menempel key langsung di kode.

In [ ]:
from google import genai
from google.genai import types

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GEMMA') or userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = os.environ.get('GEMMA') or os.environ.get('GEMINI_API_KEY')

if not api_key:
    raise RuntimeError('API key tidak ditemukan. Tambahkan Secret bernama GEMMA di Colab (ikon kunci).')

MODEL = 'gemma-4-26b-a4b-it'
client = genai.Client(api_key=api_key)
print('Google GenAI Client siap. Model:', MODEL)

## 3. Prompt (inline) & Parser JSON Toleran
* `PROMPT_V1`: prompt lama (hanya judul video) -> pembanding.
* `PROMPT_V2`: prompt baru berbasis Pedoman Anotasi v2 + konteks video. **Boleh diedit langsung di sini.**
* Render v1 = daftar datar; render v2 = dikelompokkan per video (judul/channel/deskripsi/tag ditulis sekali).
* Parser toleran: menerima `{"results":[...]}` (v1) maupun array `[{...}]` (v2), kunci `relevancy`/`relevan`.

In [ ]:
PROMPT_V1 = 'You are an expert NLP assistant specializing in energy policy in Indonesia.\nAnalyze whether each YouTube Comment is related to the topic of PLTN (nuklir / nuclear power plants) or PLTU (batu bara / coal-fired power plants) in Indonesia, contextually aided by the Video Title.\n\nAnalyze the following list:\n{{ITEMS}}\n\nReturn JSON only, in this format: {"results": [{"id": <ID>, "relevancy": true or false}]}\n'

PROMPT_V2 = 'Kamu adalah annotator untuk riset opini publik tentang PLTU (pembangkit listrik batu bara) dan PLTN (pembangkit listrik tenaga nuklir) di Indonesia.\nTugas: untuk SETIAP komentar YouTube di bawah, putuskan apakah komentar itu RELEVAN atau TIDAK menurut Pedoman Anotasi v2 berikut. Nilai tiap komentar sendiri-sendiri; jangan terpengaruh komentar lain.\n\n== DEFINISI (B1) ==\nKomentar RELEVAN hanya jika DUA syarat terpenuhi:\n1. TOPIK: ada pendapat, informasi, atau pertanyaan tentang pembangkit PLTU (batu bara) atau PLTN (nuklir) itu sendiri: dibangun, dilanjutkan, dipensiunkan, ditutup, dampaknya, biayanya, kesiapannya, kebijakannya. Termasuk harga/pasokan/kontrak/utang batu bara yang dibeli PLN untuk listrik, DMO, pembangkit swasta/IPP (B11), dan energi terbarukan yang dipakai sebagai pengganti atau pembanding PLTU/PLTN (B16).\n2. INDONESIA: konteksnya Indonesia.\nPenanda Indonesia (B2): Indonesia, RI, Indo, "kita", "kami", "negara ini", Konoha, nama daerah/pembangkit/lembaga Indonesia (PLN, BRIN, BAPETEN, BATAN, ESDM, Paiton, Suralaya, Batang, Cirebon, Babel). "Pemerintah/presiden/menteri/DPR" tanpa keterangan negara dianggap Indonesia (B3). Bahasa komentar BUKAN penanda Indonesia (B4). Membandingkan dengan negara lain tetap relevan bila tujuannya menilai Indonesia, misalnya "Jepang bisa bangun PLTN, masa kita ga bisa" (B5).\n\n== MEMAKAI INFO VIDEO (B6-B8) ==\n- Judul/deskripsi video hanya boleh dipakai untuk (a) memahami komentar yang menanggapi isi video, seperti "itu/ini", "proyek ini", "jangan", "belum saatnya", "maksudnya", dan (b) menentukan negara, selama video memang tentang PLTU/PLTN/batu bara/energi di Indonesia.\n- Komentar pendek yang jelas menanggapi topik video RELEVAN kalau videonya tentang PLTU/PLTN di Indonesia. Topik pembangkit tetap harus ada di komentar atau jelas merujuk ke topik video.\n- Video tentang negara lain (Chernobyl, Hiroshima, Jepang, Jerman, Iran) atau penjelasan umum (cara kerja PLTU/reaktor) TIDAK memberi konteks Indonesia. "Kita" di video sains umum berarti umat manusia, bukan Indonesia.\n- Tes cepat (B8): kalau komentar dipindah ke video lain dan artinya tetap sama, komentar itu TIDAK RELEVAN.\n\n== TIDAK RELEVAN ==\n- Hanya menyebut kata PLTU/PLTN tanpa pendapat/info/pertanyaan tentang pembangkitnya; debat antar-komentator; meta-komentar seperti "apa hubungannya", "salah video" (B10).\n- Reply (jenis: reply) yang maksudnya hanya bisa dipahami dari komentar induk, misalnya "betul sekali", "nah itu", "kuncinya", "bisa dibangun". Reply yang sudah menyebut PLTU/PLTN sendiri tetap relevan (B9).\n- Nuklir umum, teknis reaktor luar negeri, Chernobyl/Fukushima, senjata/rudal/bom/kapal nuklir, baterai nuklir, radiasi, limbah; reaktor riset (Serpong/BATAN/PRSG); uranium/thorium tanpa kaitan listrik (B12, B13).\n- Tambang, ekspor, harga pasar, cadangan, saham, atau bisnis batu bara/uranium/mineral tanpa kaitan dengan pembangkit (B14).\n- Kritik pemerintah, korupsi, rezim, atau politik umum yang tidak menyinggung pembangkit (B15). Kritik yang spesifik ke PLTU/PLTN atau ke kontrak/pasokannya tetap relevan.\n- Energi terbarukan/transisi energi saja tanpa menyebut PLTU/PLTN (B16).\n- Reaksi emosi ke cerita video, sapaan, pujian/terima kasih ke kreator, request konten, spam, candaan (B17).\nKomentar panjang atau banyak topik tetap RELEVAN bila topik lain dipakai untuk berargumen soal PLTU/PLTN; kata "energi" yang hanya disebut sambil lalu tidak cukup (B18).\nJangan menebak maksud tersembunyi: bila butuh asumsi panjang agar komentar jadi relevan, jawab TIDAK RELEVAN (A5).\n\n== CONTOH ==\nVideo "Pensiun dini PLTU Cirebon-1 mulai dijalankan" | "Terlalu cepat, listrik kita belum siap" -> relevan (menanggapi topik video, Indonesia) (B6)\nVideo "Rencana PLTN pertama di Indonesia" | "Semoga cepat jadi ya" -> relevan (menanggapi topik video, Indonesia) (B6)\nVideo "Rencana PLTN pertama di Indonesia" | "Kalau Jepang bisa kelola PLTN, kenapa kita tidak" -> relevan (B5)\nVideo "Chernobyl 1986, bencana nuklir terparah" | "Ngeri banget kejadiannya" -> tidak relevan (video negara lain, reaksi emosi) (B7, B17)\nVideo "Rencana PLTN pertama di Indonesia" | reply "betul sekali bang" -> tidak relevan (bergantung komentar induk) (B9)\nVideo "Ekspor batu bara naik, saham tambang menguat" | "Lagi cuan nih portofolio saya" -> tidak relevan (bisnis/saham) (B14)\nVideo "Rencana PLTN pertama di Indonesia" | "Kontennya keren, lanjutkan kak" -> tidak relevan (pujian ke kreator) (B17)\n\n{{ITEMS}}\n\n== FORMAT JAWABAN ==\nBalas HANYA dengan JSON array (tanpa teks lain, tanpa ```), satu objek per komentar, urutan sama dengan input:\n[{"id": <id>, "alasan": "<maksimal 12 kata, sebut kode aturan>", "relevan": true atau false}]\n'

def render_items(batch, style, context):
    if style == 'v1':
        return '\n'.join(f"ID: {it['i']} | Title: {it['video_title']} | Comment: {it['text']}" for it in batch)
    by_video = {}
    for it in batch:
        by_video.setdefault(it['video_id'], []).append(it)
    out = ['== VIDEO DAN KOMENTAR ==']
    for n, group in enumerate(by_video.values(), 1):
        g0 = group[0]
        out.append(f'\nVideo {n}')
        out.append(f"Judul: {g0['video_title']}")
        if context == 'full':
            if g0['video_channel']: out.append(f"Channel: {g0['video_channel']}")
            if g0['video_desc']:    out.append(f"Deskripsi: {g0['video_desc']}")
            if g0['video_tags']:    out.append(f"Tag: {g0['video_tags']}")
        out.append('Komentar:')
        for it in group:
            jenis = 'reply' if str(it['is_reply']) == '1' else 'komentar'
            out.append(f"[{it['i']}] ({jenis}) {it['text']}")
    return '\n'.join(out)

def build_prompt(template, batch, style, context):
    return template.replace('{{ITEMS}}', render_items(batch, style, context))

FENCE_RE = re.compile(r'```(?:json)?', re.I)

def _to_bool(v):
    if isinstance(v, bool): return v
    if isinstance(v, (int, float)): return bool(v)
    if isinstance(v, str):
        s = v.strip().lower()
        if s in ('true','ya','yes','relevan','1'): return True
        if s in ('false','tidak','no','tidak relevan','irrelevant','0'): return False
    return None

def parse_response(text, expected):
    """{i: (relevan_bool, alasan)} untuk id yang diminta; id tak terbaca diabaikan."""
    if not text: return {}
    tx = FENCE_RE.sub('', text)
    dec = json.JSONDecoder()
    for m in re.finditer(r'[\[{]', tx):
        try:
            data, _ = dec.raw_decode(tx[m.start():])
        except json.JSONDecodeError:
            continue
        if isinstance(data, dict):
            data = data.get('results', data.get('hasil', [data]))
        if not isinstance(data, list):
            continue
        out = {}
        for it in data:
            if not isinstance(it, dict): continue
            try: i = int(it.get('id'))
            except (TypeError, ValueError): continue
            val = _to_bool(it.get('relevan', it.get('relevancy', it.get('relevant'))))
            if i in expected and val is not None:
                out[i] = (val, str(it.get('alasan', ''))[:160])
        if out:
            return out
    return {}

print('Prompt & parser siap. Panjang PROMPT_V2:', len(PROMPT_V2), 'karakter')

## 4. Eksekusi Stateful & Resumable (3 Konfigurasi)
* `v1_original` tidak memanggil API (ambil dari kolom `v1_original` di dataset).
* `v1_title` & `v2_ctx` memanggil API dengan pacing + retry (500/429 di-retry; gagal total = `unknown`, **bukan** dianggap tidak-relevan).
* Checkpoint per config: `preds_relevan_<config>.csv`. Jalankan ulang sel ini untuk melanjutkan yang terputus.
* **Hemat kuota/waktu**: untuk run cepat, isi `CONFIGS_TO_RUN = ['v1_original', 'v2_ctx']` saja.

In [ ]:
MIN_GAP = 4.0        # jeda antar panggilan (detik); naikkan ke 8-10 bila server sering 500
MAX_RETRIES = 6

CONFIGS = {
    'v1_original': {'kind': 'baseline_col', 'col': 'v1_original'},
    'v1_title':    {'template': PROMPT_V1, 'style': 'v1', 'context': 'title', 'batch': 20},
    'v2_ctx':      {'template': PROMPT_V2, 'style': 'v2', 'context': 'full', 'batch': 8},
    # opsional (ablation prompt baru tanpa konteks video):
    # 'v2_title':  {'template': PROMPT_V2, 'style': 'v2', 'context': 'title', 'batch': 8},
}
CONFIGS_TO_RUN = ['v1_original', 'v1_title', 'v2_ctx']

ITEMS = df.to_dict('records')
EXPECTED_IDS = [int(it['i']) for it in ITEMS]

def make_batches(items, size, style):
    seq = sorted(items, key=lambda x: (x['video_id'], int(x['i']))) if style == 'v2' else items
    return [seq[k:k+size] for k in range(0, len(seq), size)]

def generate(prompt):
    cfg = types.GenerateContentConfig(temperature=0.0, response_mime_type='application/json')
    resp = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
    return resp.text or ''

def backoff(err, attempt):
    if re.search(r'429|RESOURCE_EXHAUSTED|quota', str(err), re.I): return 30 + 10 * attempt
    return min(5 * (2 ** attempt), 60)

def run_api_config(name, cfg):
    path = f'preds_relevan_{name}.csv'
    preds = {}
    if os.path.exists(path):
        prev = pd.read_csv(path, dtype=str, keep_default_na=False)
        preds = {int(r.i): (r.pred == '1', r.alasan) for r in prev.itertuples(index=False) if r.pred in ('0','1')}
        print(f'[resume] {name}: {len(preds)} sudah ada')
    todo = [it for it in ITEMS if int(it['i']) not in preds]
    batches = make_batches(todo, cfg['batch'], cfg['style'])
    print(f'[{name}] {len(ITEMS)} komentar | sisa {len(todo)} | {len(batches)} panggilan API')
    last = 0.0
    for bn, batch in enumerate(batches, 1):
        pending, attempt = list(batch), 0
        while pending and attempt <= MAX_RETRIES:
            gap = MIN_GAP - (time.time() - last)
            if gap > 0: time.sleep(gap)
            last = time.time()
            prompt = build_prompt(cfg['template'], pending, cfg['style'], cfg['context'])
            try:
                text = generate(prompt)
            except Exception as e:
                msg = str(e)
                if re.search(r'API key|API_KEY_INVALID|UNAUTHENTICATED|PERMISSION_DENIED|\b401\b|\b403\b', msg):
                    raise RuntimeError('API key ditolak/tak berizin: ' + msg[:200])
                w = backoff(e, attempt)
                print(f'    ! {name} b{bn}: {msg[:80]} -> ulang {w:.0f}s ({attempt+1}/{MAX_RETRIES})')
                time.sleep(w); attempt += 1; continue
            parsed = parse_response(text, {int(it['i']) for it in pending})
            if not parsed:
                w = backoff(RuntimeError(''), attempt)
                print(f'    ! {name} b{bn}: jawaban tak terbaca -> ulang {w:.0f}s ({attempt+1}/{MAX_RETRIES})')
                time.sleep(w); attempt += 1; continue
            preds.update(parsed)
            pending = [it for it in pending if int(it['i']) not in parsed]
            if pending: attempt += 1
        rows = [{'i': int(it['i']), 'cid': it['cid'],
                 'pred': ('' if int(it['i']) not in preds else int(preds[int(it['i'])][0])),
                 'alasan': preds.get(int(it['i']), (None, ''))[1]} for it in ITEMS]
        pd.DataFrame(rows).to_csv(path, index=False, encoding='utf-8')
        print(f'  batch {bn}/{len(batches)} selesai ({len(preds)}/{len(ITEMS)} terisi)')
    return preds

def run_baseline_col(col):
    return {int(it['i']): (str(it[col]) == '1', f'kolom {col}') for it in ITEMS if str(it[col]) in ('0','1')}

ALL_PREDS = {}
for name in CONFIGS_TO_RUN:
    cfg = CONFIGS[name]
    ALL_PREDS[name] = run_baseline_col(cfg['col']) if cfg.get('kind') == 'baseline_col' else run_api_config(name, cfg)
print('\nSemua konfigurasi selesai.')

## 5. Evaluasi Metrik Biner & Analisis Error
* **Set gold 80% relevan** -> gunakan **recall** & **specificity**, bukan akurasi mentah.
* **prec@20% / prec@30%** = perkiraan presisi bila di data asli hanya 20%/30% komentar relevan.
* `unknown` = gagal diproses setelah semua retry; **tidak** dihitung sebagai salah.
* `errors_relevan_<config>.csv` = komentar salah (FN/FP) + alasan model, untuk audit annotator.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return float('nan'), float('nan')
    p, d = k / n, 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c-h, c+h

def binary_metrics(name, preds):
    ids = [i for i in EXPECTED_IDS if i in preds]
    unknown = len(EXPECTED_IDS) - len(ids)
    tp = sum(1 for i in ids if y_true[i] and preds[i][0])
    fn = sum(1 for i in ids if y_true[i] and not preds[i][0])
    fp = sum(1 for i in ids if not y_true[i] and preds[i][0])
    tn = sum(1 for i in ids if not y_true[i] and not preds[i][0])
    n = tp+fn+fp+tn
    div = lambda a, b: a/b if b else float('nan')
    rec, spec, prec = div(tp, tp+fn), div(tn, tn+fp), div(tp, tp+fp)
    row = {'config': name, 'n': n, 'unknown': unknown, 'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
           'accuracy': div(tp+tn, n), 'recall': rec, 'specificity': spec, 'precision': prec,
           'F1': div(2*tp, 2*tp+fp+fn)}
    lo, hi = wilson(tp, tp+fn); row['recall_95CI'] = f'{lo:.0%}-{hi:.0%}'
    lo, hi = wilson(tn, tn+fp); row['spec_95CI'] = f'{lo:.0%}-{hi:.0%}'
    for pi in (0.2, 0.3):
        row[f'prec@{int(pi*100)}%'] = div(rec*pi, rec*pi + (1-spec)*(1-pi)) if rec==rec and spec==spec else float('nan')
    return row

summary = [binary_metrics(name, preds) for name, preds in ALL_PREDS.items()]
summary_df = pd.DataFrame(summary)
pct = ['accuracy','recall','specificity','precision','F1','prec@20%','prec@30%']
print('=' * 95)
print('HASIL RELEVANSI pada', len(EXPECTED_IDS), 'komentar gold (80% relevan) | model', MODEL)
print('=' * 95)
display(summary_df.style.format({c: '{:.1%}' for c in pct}))

# dump error per config
byid = {int(it['i']): it for it in ITEMS}
for name, preds in ALL_PREDS.items():
    ids = [i for i in EXPECTED_IDS if i in preds]
    wrong = [{'jenis': 'FN' if y_true[i] else 'FP', 'cid': byid[i]['cid'], 'target': byid[i]['target'],
              'video_title': byid[i]['video_title'], 'text': byid[i]['text'],
              'gold_relevan': int(y_true[i]), 'pred': int(preds[i][0]), 'alasan': preds[i][1]}
             for i in ids if preds[i][0] != y_true[i]]
    pd.DataFrame(wrong).to_csv(f'errors_relevan_{name}.csv', index=False, encoding='utf-8')
    print(f'{name}: {len(wrong)} salah -> errors_relevan_{name}.csv')

summary_df.to_csv('summary_relevan_latest.csv', index=False, encoding='utf-8')
print('\nRingkasan disimpan: summary_relevan_latest.csv')
print('Catatan: set gold 80% relevan; gunakan recall/specificity & prec@20%/30%, bukan akurasi mentah.')

## 6. Flowchart Metodologi Relevansi (Mermaid Diagram)

In [ ]:
import base64
from IPython.display import Image, display

def render_mermaid(graph):
    b64 = base64.b64encode(graph.encode('ascii')).decode('ascii')
    display(Image(url='https://mermaid.ink/img/' + b64))

render_mermaid('''
graph TD
    A[Muat relevance_input_yt.csv<br/>124 komentar, 80% relevan]:::s
    C{Konfigurasi}:::f
    O[v1_original: kolom gold<br/>0 panggilan API]:::p
    V1[v1_title: prompt lama<br/>judul saja]:::a
    V2[v2_ctx: prompt Pedoman v2<br/>+ konteks video penuh]:::a
    R[Batch + pacing + retry<br/>checkpoint resumable]:::p
    E[Metrik biner: recall, specificity,<br/>prec@20/30, Wilson 95%CI]:::v
    A --> C
    C --> O --> E
    C --> V1 --> R
    C --> V2 --> R --> E
    classDef s fill:#2b2d42,color:#fff,stroke:#1d1e2c,rx:6px;
    classDef f fill:#f39c12,color:#fff,stroke:#d68910,rx:6px;
    classDef p fill:#457b9d,color:#fff,stroke:#1d3557,rx:6px;
    classDef a fill:#3a86ff,color:#fff,stroke:#004e92,rx:6px;
    classDef v fill:#06d6a0,color:#fff,stroke:#0f8b6d,rx:6px;
''')